# MyTravelHelper: Streamlit + Hugging Face Inference Providers

Notebook này trình bày quy trình xây dựng ứng dụng trợ lý du lịch MyTravelHelper theo yêu cầu bài tập: mục tiêu, phân tích yêu cầu, pipeline, thiết lập môi trường, lựa chọn model, test cơ bản và các chức năng nâng cao.

## 1. Mục tiêu và phân tích yêu cầu

**Mục tiêu:** xây dựng ứng dụng trợ lý du lịch bằng Streamlit, dùng Hugging Face Inference Providers để gọi các mô hình NLP/LLM mà không cần chạy model nặng trên máy local.

**Chức năng chính:**

- Chatbot tư vấn du lịch.
- Phân loại ý định và trích xuất thực thể từ yêu cầu người dùng.
- Phân tích cảm xúc review du lịch theo khía cạnh.
- Phát hiện các chủ đề nổi bật trong nhiều review.

**Ràng buộc:** app vẫn nên demo được khi chưa có token Hugging Face, nên phần code có fallback heuristic cục bộ.

## 2. Kiến trúc tổng quát

```mermaid
flowchart LR
    A[Streamlit Input] --> B[Intent + Entity Extraction]
    B --> C{Routing}
    C --> D[LLM Travel Chat]
    C --> E[Aspect Sentiment]
    C --> F[Topic Detection]
    D --> G[Streamlit Output]
    E --> G
    F --> G
```

Pipeline gồm 5 lớp: input, intent/NER, routing, processing modules và output. Cách tổ chức này giúp tách giao diện Streamlit khỏi logic xử lý NLP.

## 3. Thiết lập môi trường với micromamba

```bash
micromamba create -n mytravelhelper python=3.11
micromamba activate mytravelhelper
pip install -r requirements.txt
```

Thiết lập token Hugging Face trong file `.env`:

```bash
HF_TOKEN=hf_your_token_here
```

Nếu chưa có `HF_TOKEN`, app vẫn chạy ở chế độ fallback để kiểm thử luồng xử lý.

## 4. Model sử dụng

| Nhiệm vụ | Model dự kiến | Vai trò |
|---|---|---|
| Text generation / Chat | `mistralai/Mistral-7B-Instruct-v0.2` | Sinh phản hồi tư vấn du lịch |
| Intent classification | `facebook/bart-large-mnli` | Zero-shot intent filtering |
| NER | `dslim/bert-base-NER` | Nhận diện thực thể |
| Sentiment | `cardiffnlp/twitter-roberta-base-sentiment-latest` | Hỗ trợ phân tích cảm xúc |

Các model được gọi qua `huggingface_hub.InferenceClient` trong module `modules/inference.py`.

In [ ]:
from modules.inference import HuggingFaceService

hf = HuggingFaceService()
hf.healthcheck()

## 5. Test phân loại ý định và trích xuất thực thể

In [ ]:
from modules.nlp_tasks import INTENT_LABELS, extract_intent_and_entities

query = "Mình muốn đi Hội An 3 ngày với ngân sách 4 triệu, nên tham quan ở đâu?"
extract_intent_and_entities(
    query,
    hf.zero_shot_intent(query, INTENT_LABELS),
    hf.named_entities(query),
)

## 6. Test phân tích cảm xúc theo khía cạnh

In [ ]:
from modules.nlp_tasks import analyze_sentiment_aspects

review = "Khách sạn gần biển, phòng sạch và view đẹp. Nhân viên thân thiện nhưng bữa sáng ít món. Giá đáng tiền."
analyze_sentiment_aspects(review)

## 7. Test phát hiện chủ đề

In [ ]:
from modules.nlp_tasks import detect_topics

reviews = [
    "Biển đẹp, hải sản ngon, giá hơi cao.",
    "Khách sạn gần trung tâm, phòng sạch, lễ tân hỗ trợ nhanh.",
    "Di chuyển từ sân bay tiện, nhưng vé tham quan khá đắt.",
]
detect_topics(reviews)

## 8. Chạy ứng dụng Streamlit

```bash
streamlit run app.py
```

Các tab trong app tương ứng với yêu cầu: chatbot, phân tích review, intent/NER, topic detection và mô tả pipeline.